In [ ]:
# Importação das bibliotecas utilizadas

import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

from scipy.io import arff

from scipy.stats import chi2_contingency

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import ( accuracy_score, classification_report, confusion_matrix, roc_curve, auc )

In [ ]:
!wget -O asd_children.zip "https://archive.ics.uci.edu/static/public/419/autistic+spectrum+disorder+screening+data+for+children.zip"
!unzip -o asd_children.zip

# Download e carregamento da base pública do UCI
data, meta = arff.loadarff('Autism-Child-Data.arff')
df = pd.DataFrame(data)

# Conversão de variáveis categóricas (bytes para string)
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.decode('utf-8')

df.head()

In [ ]:
# Conversão da variável alvo para formato binário

df['Class/ASD'] = df['Class/ASD'].map({'NO': 0, 'YES': 1})
df['Class/ASD'].value_counts()

In [ ]:
plt.figure(figsize=(6,4))
df["Class/ASD"].value_counts().sort_index().plot(kind="bar", color="black")

plt.xlabel("Classe (0 = ausência, 1 = presença)")
plt.ylabel("Frequência")

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)
ax.grid(False)

plt.tight_layout()
plt.savefig("fig1.png", dpi=300)
plt.show()

In [ ]:
# Separação entre variáveis explicativas e variável alvo

X = df.drop("Class/ASD", axis=1)
y = df["Class/ASD"]

# Transformação de variáveis categóricas (one-hot encoding)

X = pd.get_dummies(X, drop_first=True)
X = X.drop("result", axis=1, errors="ignore")
# Remoção da coluna redundante 'result', quando presente
X = X.fillna(0)

# Divisão em treino (70%) e teste (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

In [ ]:
# Treinamento da Regressão Logística

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predições e probabilidades

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Accuracy Logística:", model.score(X_test, y_test))
print(classification_report(y_test, y_pred))

In [ ]:
# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(4,4))
ax = plt.gca()
ax.set_facecolor("white")
plt.gcf().patch.set_facecolor("white")
plt.imshow(np.ones_like(cm), cmap="gray", vmin=0, vmax=1)
for i in range(3):
    ax.axhline(i-0.5, color='black', linewidth=1.5)
    ax.axvline(i-0.5, color='black', linewidth=1.5)
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j],
                 ha="center", va="center",
                 color="black")

plt.xticks([0,1], ["0","1"])
plt.yticks([0,1], ["0","1"])
plt.xlabel("Previsto")
plt.ylabel("Real")

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

ax.grid(False)

plt.tight_layout()
plt.savefig("fig2.png", dpi=300, facecolor="white")
plt.show()

In [ ]:

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob)
auc_lr = auc(fpr_lr, tpr_lr)

plt.figure(figsize=(6,4))
plt.plot(fpr_lr, tpr_lr, color="black", linewidth=2)
plt.plot([0,1],[0,1], linestyle="--", color="black")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)
ax.grid(False)

plt.tight_layout()
plt.savefig("fig3.png", dpi=300)
plt.show()

print("AUC Logística:", auc_lr)

In [ ]:
coef = model.coef_[0]
features = X.columns

coef_df = pd.DataFrame({
    "Variável": features,
    "Coeficiente": coef
})

coef_df["Importância"] = np.abs(coef_df["Coeficiente"])
coef_df = coef_df.sort_values("Importância", ascending=False)

top = coef_df.head(10)

plt.figure(figsize=(6,4))
plt.barh(top["Variável"], top["Coeficiente"], color="black")

plt.xlabel("Coeficiente")
plt.gca().invert_yaxis()

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)
ax.grid(False)

plt.tight_layout()
plt.savefig("fig4.png", dpi=300)
plt.show()

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:,1]

print("Accuracy Random Forest:", rf.score(X_test, y_test))
print(classification_report(y_test, rf_pred))

In [ ]:
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_prob)
auc_rf = auc(fpr_rf, tpr_rf)

plt.figure(figsize=(6,4))

plt.plot(fpr_lr, tpr_lr, color="black", linewidth=2, linestyle="-")
plt.plot(fpr_rf, tpr_rf, color="black", linewidth=2, linestyle="--")
plt.plot([0,1],[0,1], linestyle=":", color="black")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)
ax.grid(False)

plt.legend([
    f"Logística (AUC = {auc_lr:.2f})",
    f"Random Forest (AUC = {auc_rf:.2f})"
], frameon=False)

plt.tight_layout()
plt.savefig("fig5.png", dpi=300)
plt.show()

In [ ]:
tab = pd.crosstab(df["gender"], df["Class/ASD"], normalize="index")

plt.style.use("default")
plt.figure(figsize=(6,4))

ax = plt.gca()
ax.set_facecolor("white")
plt.gcf().patch.set_facecolor("white")
tab.plot(kind="bar",
         stacked=True,
         color=["black","white"],
         edgecolor="black",
         linewidth=1.2,
         ax=ax)

plt.xlabel("Sexo")
plt.ylabel("Proporção")
plt.xticks(rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)
ax.spines['left'].set_color("black")
ax.spines['bottom'].set_color("black")

ax.grid(False)
plt.legend(frameon=False)

plt.tight_layout()
plt.savefig("fig6.png", dpi=300, facecolor="white")
plt.show()

In [ ]:
# Teste Qui-quadrado para verificar associação entre sexo e classificação

from scipy.stats import chi2_contingency


tab_abs = pd.crosstab(df["gender"], df["Class/ASD"])

chi2, p, dof, expected = chi2_contingency(tab_abs)

print("Qui-quadrado:", chi2)
print("p-valor:", p)

In [ ]:
# Construção da tabela de contingência absoluta entre
# histórico familiar ("austim") e classificação final ("Class/ASD")
tab_hist = pd.crosstab(df["austim"], df["Class/ASD"])

# Construção da tabela proporcional por linha (percentual dentro de cada grupo)
# O parâmetro normalize="index" calcula proporções relativas ao total de cada categoria de histórico familiar
tab_hist_pct = pd.crosstab(
    df["austim"],
    df["Class/ASD"],
    normalize="index"
) * 100

tab_hist
tab_hist_pct

In [ ]:
# Teste Qui-quadrado de independência entre histórico familiar e classificação final

chi2, p, dof, expected = chi2_contingency(tab_hist)


print("Qui-quadrado:", chi2)
print("p-valor:", p)

In [ ]:
# Construção de um novo conjunto de variáveis explicativas
# removendo qualquer coluna relacionada ao histórico familiar
X_sem_hist = X.drop(
    columns=[col for col in X.columns if "austim" in col]
)

# Nova divisão em treino e teste, mantendo os mesmos parâmetros
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_sem_hist,
    y,
    test_size=0.3,
    random_state=42
)

# Treinamento de nova Regressão Logística sem histórico familiar
model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train2, y_train2)

# Cálculo das probabilidades previstas
y_prob2 = model2.predict_proba(X_test2)[:, 1]

# Cálculo da curva ROC e da AUC
fpr2, tpr2, _ = roc_curve(y_test2, y_prob2)
auc2 = auc(fpr2, tpr2)

# Comparação entre os modelos
print("AUC sem histórico:", auc2)
print("AUC com histórico:", auc_lr)

In [ ]:
tab_hist_pct = pd.crosstab(
    df["austim"],
    df["Class/ASD"],
    normalize="index"
) * 100

fig, ax = plt.subplots(figsize=(6,4))

tab_hist_pct.plot(
    kind="bar",
    stacked=True,
    color=["black", "white"],
    edgecolor="black",
    ax=ax
)

ax.set_xlabel("Histórico Familiar")
ax.set_ylabel("Proporção (%)")
ax.set_xticklabels(["Não", "Sim"], rotation=0)

ax.set_facecolor("white")
fig.patch.set_facecolor("white")

ax.grid(False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

plt.tight_layout()
plt.savefig("fig_hist_familiar.png", dpi=300, facecolor="white")
plt.show()

In [ ]:
# Selecionar apenas as 10 variáveis do AQ-10
aq10_cols = [col for col in df.columns if "A" in col and "_Score" in col]

X_aq10 = df[aq10_cols]
y = df["Class/ASD"]

# Separação treino/teste
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_aq10, y, test_size=0.3, random_state=42
)

# Treinar modelo logístico simplificado
from sklearn.linear_model import LogisticRegression

model_aq10 = LogisticRegression(max_iter=1000)
model_aq10.fit(X_train, y_train)

print("Accuracy (Modelo Simplificado):", model_aq10.score(X_test, y_test))

In [ ]:
coeficientes_reais = model_aq10.coef_[0]
intercept_real = model_aq10.intercept_[0]

coef_df_real = pd.DataFrame({
    "Variável": aq10_cols,
    "Coeficiente": coeficientes_reais
})

coef_df_real

In [ ]:
print(model_aq10.intercept_[0])

In [ ]:
coeficientes = [
    1.783907,
    1.368032,
    1.470703,
    1.935939,
    2.044535,
    1.228392,
    1.412613,
    1.737710,
    1.583022,
    2.052199
]

intercept = -10.672538874097201

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import math

perguntas = [
    "A1. Foca mais em detalhes do que no todo?",
    "A2. Dificuldade em entender o ponto de vista alheio?",
    "A3. Prefere fazer as coisas da mesma forma sempre?",
    "A4. Gosta de planejar atividades cuidadosamente?",
    "A5. Fica muito focado em padrões e números?",
    "A6. Nota detalhes que os outros não percebem?",
    "A7. Dificuldade em saber as intenções dos outros?",
    "A8. Dificuldade em ler as entrelinhas?",
    "A9. Fica ansioso em situações sociais novas?",
    "A10. Dificuldade em fazer novos amigos?"
]

seletores = []
for p in perguntas:
    seletores.append(
        widgets.Dropdown(
            options=['Não', 'Sim'],
            value='Não',
            layout=widgets.Layout(width='100px')
        )
    )

titulo = HTML("<h2>Termômetro de Triagem: Indícios de TEA (Regressão Logística)</h2><hr>")
botao = widgets.Button(description="Processar Avaliação", button_style='info', icon='stethoscope')
saida = widgets.Output()

coluna_esq = widgets.VBox([
    widgets.HBox([widgets.Label(perguntas[i], layout=widgets.Layout(width='350px')), seletores[i]])
    for i in range(5)
])

coluna_dir = widgets.VBox([
    widgets.HBox([widgets.Label(perguntas[i], layout=widgets.Layout(width='350px')), seletores[i]])
    for i in range(5, 10)
])

formulario = widgets.HBox([coluna_esq, coluna_dir])

def calcular_risco(b):
    with saida:
        clear_output()

        respostas_num = [1 if s.value == 'Sim' else 0 for s in seletores]

        coeficientes = [
            1.783907,
            1.368032,
            1.470703,
            1.935939,
            2.044535,
            1.228392,
            1.412613,
            1.737710,
            1.583022,
            2.052199
        ]

        intercept = -10.672538874097201

        pontuacao_total = intercept + sum(r * c for r, c in zip(respostas_num, coeficientes))
        probabilidade = 1 / (1 + math.exp(-pontuacao_total))
        prob_percentual = int(probabilidade * 100)

        if prob_percentual < 40:
            cor = "#28a745"
            mensagem = "<h3 style='color: #28a745;'>[Risco Baixo] Sinais mínimos. Acompanhamento padrão.</h3>"
        elif prob_percentual < 70:
            cor = "#ffc107"
            mensagem = "<h3 style='color: #d39e00;'>[Risco Moderado] Sinais presentes. Observação recomendada.</h3>"
        else:
            cor = "#dc3545"
            mensagem = "<h3 style='color: #dc3545;'>[ALERTA MÉDICO] Sugere-se encaminhamento prioritário para especialista!</h3>"

        html_resultado = f"""
        <br>
        <div style="width: 80%; background-color: #e0e0e0; border-radius: 10px; border: 1px solid #ccc;">
          <div style="width: {prob_percentual}%; background-color: {cor}; padding: 10px 0; color: white; text-align: center; border-radius: 10px; font-weight: bold; font-size: 16px;">
            {prob_percentual}%
          </div>
        </div>
        {mensagem}
        """

        display(HTML(html_resultado))

botao.on_click(calcular_risco)

display(titulo, formulario, HTML("<br>"), botao, saida)